# Week 2: regression of house prices (California housing)

*Deep Learning for Engineering, Durham University, Michaelmas Term 2026*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimageramiseresht/ENGI-48915---Deep-Learning-for-Engineering/blob/main/week2/housing_regression.ipynb)

Predict the **median house value** of a California district from 8 numbers describing it: location, house age, size, population and income. This notebook follows the regression slides in Sections 2 and 6 of the lecture.

This is the engineering situation in miniature: **a continuous target and very little data**, only 480 training samples. Small data changes how we validate (K-fold), how big the model can be, and how easily it overfits.

| | MNIST | Housing |
|---|---|---|
| target | a class, 0 to 9 | a price in dollars |
| last layer | `Dense(10, activation="softmax")` | `Dense(1)`, no activation |
| loss | `sparse_categorical_crossentropy` | `mean_squared_error` |
| metric | accuracy | mean absolute error, in target units |
| validation | one 10% split | K-fold cross-validation |

> **Why not the Boston housing dataset?** It is the classic version of this exercise, but it contains a feature built on an assumption about race. Keras now strongly discourages using it, and provides this small California version as its replacement.

Run the cells from top to bottom with **Shift + Enter**. Everything runs on a laptop CPU in a few minutes.

## Step 0: setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(42)
print(keras.__version__, keras.backend.backend())

## Step 1: load the data and look at it

The small version has 600 districts: 480 for training and 120 for testing.

In [ ]:
(xr_train, yr_train), (xr_test, yr_test) = \
    keras.datasets.california_housing.load_data(version="small")
print(xr_train.shape, xr_test.shape)   # (480, 8) (120, 8)

print("feature        min         max        mean")
for j in range(xr_train.shape[1]):
    col = xr_train[:, j]
    print(f"{j:>7} {col.min():>11.2f} {col.max():>11.2f} {col.mean():>11.2f}")
print(f"target: ${yr_train.min():,.0f} to ${yr_train.max():,.0f}")

plt.figure(figsize=(7, 3))
plt.hist(yr_train, bins=30)
plt.xlabel("median house value ($)"); plt.ylabel("districts")
plt.show()

**Question:** look at the feature ranges. Why would they cause trouble if fed to the network as they are?

<details><summary>Show the answer</summary>

Some features span a few units, others tens of thousands. With raw inputs, the large-scale features dominate the gradients and stretch the loss surface, so one learning rate cannot suit every direction (Section 4). The initial weights also assume inputs of order 1.

</details>

## Step 2: preprocessing

- **Standardise each feature** with the training mean and standard deviation, then apply the *same* numbers to the test set. Computing them on all the data would leak test information into training.
- **Scale the target** to a few units: with small initial weights, the network's first outputs are near 0, and reaching 500,000 would need huge weights and slow training. Multiply predictions by 100,000 to get dollars back.

In [ ]:
mean = xr_train.mean(axis=0)     # training statistics only
std = xr_train.std(axis=0)
xr_train = (xr_train - mean) / std
xr_test = (xr_test - mean) / std  # the same mean and std

yr_train = yr_train / 100_000    # dollars become units of $100k
yr_test = yr_test / 100_000

print(np.round(xr_train.mean(axis=0), 2))   # all about 0
print(np.round(xr_train.std(axis=0), 2))    # all 1
print(np.round(xr_test.mean(axis=0), 2))    # close to 0, not exactly: that is expected

## Step 3: build the model

Two hidden layers of 64 units, and **one output unit with no activation**: a price can take any value (Section 2). The network is small on purpose: with 480 samples, a large model memorises the training set quickly.

Train with **MSE**; report **MAE**, which reads directly in target units: an MAE of 0.30 means about $30,000 off on average.

**Question:** how many parameters?

<details><summary>Show the answer</summary>

8 × 64 + 64 = 576, then 64 × 64 + 64 = 4,160, then 64 + 1 = 65: **4,801** in total, about ten per training sample.

</details>

In [ ]:
def build_regressor():
    model = keras.Sequential([
        keras.Input(shape=(8,)),           # was 784
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),                   # no activation
    ])
    model.compile(optimizer="adam",
                  loss="mean_squared_error",
                  metrics=["mean_absolute_error"])
    return model

build_regressor().summary()

## Step 4: K-fold cross-validation

A single 20% validation split would hold about 100 samples, and its score would depend heavily on *which* 100. Instead, split the training data into *K* = 4 folds, train 4 fresh models, each validated on a different fold, and average. Every sample is used for validation exactly once.

In [ ]:
k = 4
n = len(xr_train) // k
scores = []
for i in range(k):
    val = slice(i * n, (i + 1) * n)
    xv, yv = xr_train[val], yr_train[val]
    xf = np.concatenate([xr_train[:i * n],
                         xr_train[(i + 1) * n:]])
    yf = np.concatenate([yr_train[:i * n],
                         yr_train[(i + 1) * n:]])
    m = build_regressor()
    m.fit(xf, yf, epochs=50, batch_size=16, verbose=0)
    scores.append(m.evaluate(xv, yv, verbose=0)[1])
print("fold MAE:", np.round(scores, 3))
print("mean MAE:", np.mean(scores))

**Is that good?** Compare with the simplest possible model: predict the training mean for every district, fold by fold.

In [ ]:
baseline = []
for i in range(k):
    val = slice(i * n, (i + 1) * n)
    train_mean = np.concatenate([yr_train[:i * n], yr_train[(i + 1) * n:]]).mean()
    baseline.append(np.mean(np.abs(yr_train[val] - train_mean)))
print("baseline fold MAE:", np.round(baseline, 3))
print(f"baseline mean MAE: {np.mean(baseline):.3f}  (about ${np.mean(baseline) * 100_000:,.0f})")
print(f"network  mean MAE: {np.mean(scores):.3f}  (about ${np.mean(scores) * 100_000:,.0f})")

**Question:** why do the fold scores differ so much?

<details><summary>Show the answer</summary>

Each fold validates on about 120 districts. With so few, the score depends on which districts happen to land in the fold: a fold with a few unusual, expensive districts scores worse. One reference run in the lecture ranged from 0.23 to 0.35. That spread is exactly why we average over folds rather than trust one split.

</details>

## Step 5: how many epochs?

50 epochs was a guess. Record the validation MAE after every epoch in every fold, average the four curves, and find the minimum. This takes a minute or two.

In [ ]:
EPOCHS = 150
curves = []
for i in range(k):
    val = slice(i * n, (i + 1) * n)
    xf = np.concatenate([xr_train[:i * n], xr_train[(i + 1) * n:]])
    yf = np.concatenate([yr_train[:i * n], yr_train[(i + 1) * n:]])
    keras.utils.set_random_seed(i)
    h = build_regressor().fit(xf, yf, epochs=EPOCHS, batch_size=16,
                              validation_data=(xr_train[val], yr_train[val]), verbose=0)
    curves.append(h.history["val_mean_absolute_error"])

avg = np.mean(curves, axis=0)
best_epoch = int(np.argmin(avg)) + 1
print(f"lowest average validation MAE {avg.min():.3f} at epoch {best_epoch}")

plt.figure(figsize=(8, 4))
for c in curves:
    plt.plot(range(1, EPOCHS + 1), c, color="grey", alpha=0.35, lw=1)
plt.plot(range(1, EPOCHS + 1), avg, lw=2.2, label="average of the 4 folds")
plt.axvline(best_epoch, color="black", lw=0.8)
plt.ylim(0, np.percentile(avg, 98) * 1.6)
plt.xlabel("epoch"); plt.ylabel("validation MAE (units of $100k)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Question:** what do the grey curves tell you that the average hides?

<details><summary>Show the answer</summary>

How noisy a single validation fold is. One fold's curve can suggest a very different best epoch from another's. The average is much smoother, which makes the choice of epoch more trustworthy.

</details>

## Step 6: final model, and the test set once

Train one model on **all 480** training samples for `best_epoch` epochs, then evaluate on the 120 test districts, once.

In [ ]:
keras.utils.set_random_seed(0)
final = build_regressor()
final.fit(xr_train, yr_train, epochs=best_epoch, batch_size=16, verbose=0)
test_mse, test_mae = final.evaluate(xr_test, yr_test, verbose=0)
print(f"test MAE {test_mae:.3f}: about ${test_mae * 100_000:,.0f} off on average")
print(f"test RMSE {np.sqrt(test_mse):.3f}: about ${np.sqrt(test_mse) * 100_000:,.0f}")

**Question:** why is RMSE larger than MAE?

<details><summary>Show the answer</summary>

RMSE squares each error before averaging, so a few large errors weigh much more than many small ones (Section 2). RMSE is always at least as large as MAE; a big gap between them means a few districts are predicted very badly.

</details>

## Step 7: look at the predictions

In [ ]:
pred = final.predict(xr_test, verbose=0)[:, 0] * 100_000
true = yr_test * 100_000
err = pred - true

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.scatter(true, pred, s=14, alpha=0.7)
lim = [min(true.min(), pred.min()), max(true.max(), pred.max())]
ax1.plot(lim, lim, "k--", lw=1)
ax1.set(xlabel="true value ($)", ylabel="predicted value ($)", title="Test districts")
ax2.hist(err, bins=30)
ax2.set(xlabel="prediction error ($)", ylabel="districts", title="Errors")
plt.tight_layout(); plt.show()

worst = np.argsort(-np.abs(err))[:5]
for i in worst:
    print(f"true ${true[i]:>9,.0f}   predicted ${pred[i]:>9,.0f}   off by ${err[i]:>+10,.0f}")

**Question:** look at the most expensive districts in the scatter plot. What do you notice?

<details><summary>Show the answer</summary>

The points usually fall below the diagonal at the top end: expensive districts tend to be underpredicted. Few training samples are that expensive, so the model has little to learn from there, and in the original census data values are also capped at about $500,000. Errors are rarely uniform across the range: always check where a regression model is wrong, not just its average error.

</details>

---
# Part 2: overfitting with small data (Sections 7 and 8)

With 480 samples, model size matters a lot. Hold back the last 120 training samples for validation, and compare the 64-unit model with a 512-unit one. Then add dropout and early stopping to the large one.

In [ ]:
def build_sized(units, dropout=0.0):
    model = keras.Sequential([keras.Input(shape=(8,))])
    for _ in range(2):
        model.add(layers.Dense(units, activation="relu"))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    model.compile(optimizer="adam", loss="mean_squared_error",
                  metrics=["mean_absolute_error"])
    return model

x_fit, y_fit = xr_train[:360], yr_train[:360]
x_hold, y_hold = xr_train[360:], yr_train[360:]

runs = {}
for name, units, drop, cb in [("64 units", 64, 0.0, []),
                              ("512 units", 512, 0.0, []),
                              ("512 units, dropout + early stopping", 512, 0.3,
                               [keras.callbacks.EarlyStopping(monitor="val_loss", patience=20,
                                                              restore_best_weights=True)])]:
    keras.utils.set_random_seed(0)
    h = build_sized(units, drop).fit(x_fit, y_fit, epochs=300, batch_size=16,
                                     validation_data=(x_hold, y_hold),
                                     callbacks=cb, verbose=0).history
    runs[name] = h
    b = int(np.argmin(h["val_mean_absolute_error"]))
    print(f"{name:38s} best validation MAE {h['val_mean_absolute_error'][b]:.3f} at epoch {b + 1}, "
          f"final training MAE {h['mean_absolute_error'][-1]:.3f}")

plt.figure(figsize=(8, 4))
for name, h in runs.items():
    plt.plot(h["val_mean_absolute_error"], label=name)
plt.ylim(0, 1)
plt.xlabel("epoch"); plt.ylabel("validation MAE (units of $100k)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Question:** compare the final training MAE with the best validation MAE for the 512-unit model. What does the gap tell you?

<details><summary>Show the answer</summary>

Its training error keeps falling far below its validation error: with about 270,000 weights for 360 samples it can memorise the training districts, and its validation error drifts up as it does. Dropout and early stopping keep the best weights from before that happens. One 120-sample validation split is noisy, so repeat with other seeds, or with K-fold, before drawing firm conclusions.

</details>

## Exercises

1. **MAE as the loss:** train with `loss="mean_absolute_error"` in the K-fold loop. MSE learns the conditional mean, MAE the median (Section 2). Which gives the lower validation MAE here?
2. **Skip the scaling:** rerun Steps 4 to 6 without standardising the features, then without scaling the target. What happens to training?
3. **K:** try *K* = 3 and *K* = 8. How do the mean and the spread of the fold scores change, and what does each choice cost?
4. **The large version:** load `version="large"` (20,640 districts) and use a simple 20% validation split. Does the 512-unit model still overfit as badly?